# NB_00_RP_47_SOURCE_EXTRACTION

This notebook reviews the remaining Becker presentation material and converts its technology-to-market statements into structured, reviewable RP_47 inputs and cumulative YAML specifications.

```text
Detector progress
        ↓
General Atomics project plus-up
        ↓
Commercial TES opportunity evaluation
        ↓
Product assembly requirements
        ↓
RP_47_A.yaml · RP_47_B.yaml · RP_47_C.yaml
```

This notebook is grounded in Dan Becker's presentation:

> *Achieving 1% Assay of Special Nuclear Materials in 2 Minutes with Microcalorimeter-Array Gamma-Ray Spectroscopy*  
> ARPA-E Fission Annual Meeting, October 1–2, 2025.

The extraction is limited to statements supported by presentation pages 14–16. It does not infer a manufacturing partner, a completed commercial agreement, or routine operational deployment.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import json
import zipfile

try:
    import yaml
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pyyaml"],
        check=True,
    )
    import yaml

try:
    import pandas as pd
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pandas"],
        check=True,
    )
    import pandas as pd

NOTEBOOK_ID = "NB_00_RP_47_SOURCE_EXTRACTION"
NOTEBOOK_VERSION = "1.0.0"
REPOSITORY = "sensors-becker"

OUTPUT_DIRECTORY = Path("outputs/source_extraction/becker_2025_rp_47")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": REPOSITORY,
    "output_directory": str(OUTPUT_DIRECTORY),
}


## Source Identity

`source_file` records the reviewed PDF filename for provenance. This notebook does not open or parse the PDF during execution.


In [ ]:
SOURCE = {
    "source_id": "BECKER_2025_ARPA_E_MICROCALORIMETER_ASSAY",
    "title": (
        "Achieving 1% Assay of Special Nuclear Materials in 2 Minutes "
        "with Microcalorimeter-Array Gamma-Ray Spectroscopy"
    ),
    "author": "Dan Becker",
    "organization": "University of Colorado",
    "event": "ARPA-E Fission Annual Meeting",
    "date": "2025-10-01/2025-10-02",
    "source_file": "Daniel Becker (1).pdf",
    "page_count": 16,
    "reviewed_pages": [14, 15, 16],
    "engineering_object": "Microcalorimeter Technology Transition",
    "engineering_direction": (
        "Toward source-supported project participation, commercial evaluation, "
        "and product-assembly specification."
    ),
}

SOURCE


## Extraction Schema

The RP_47 extraction categories are:

```text
commercialization_indicator
project_participation
technical_scope
business_evaluation
product_requirement
deployment_scope
```


In [ ]:
@dataclass(frozen=True)
class SourceExtraction:
    extraction_id: str
    category: str
    statement: str
    page: int
    metric: str | None = None
    value: float | int | str | None = None
    unit: str | None = None
    comparison_state: str | None = None
    note: str = ""

    def validate(self) -> None:
        allowed = {
            "commercialization_indicator",
            "project_participation",
            "technical_scope",
            "business_evaluation",
            "product_requirement",
            "deployment_scope",
        }
        if self.category not in allowed:
            raise ValueError(f"Unsupported category: {self.category}")
        if self.page not in SOURCE["reviewed_pages"]:
            raise ValueError(
                f"RP_47 extraction page {self.page} is outside reviewed pages "
                f"{SOURCE['reviewed_pages']}"
            )
        if not self.statement.strip():
            raise ValueError("statement is required")


## Source-Derived Extractions

The entries below preserve the presentation's distinction between:

- detector progress that aids commercialization;
- General Atomics' interest in joining through a project plus-up;
- technical and technology-to-market scopes;
- product-assembly requirements;
- deployment objectives already carried by the project.


In [ ]:
EXTRACTIONS = [
    SourceExtraction(
        extraction_id="CI_001",
        category="commercialization_indicator",
        statement="Even 140 counts per second per detector aids commercialization.",
        page=14,
        metric="per_detector_count_rate",
        value=140,
        unit="counts_per_second",
        comparison_state="progress_indicator",
    ),
    SourceExtraction(
        extraction_id="CI_002",
        category="commercialization_indicator",
        statement="The project is very close to the target detector speed.",
        page=14,
        comparison_state="progress_indicator",
    ),
    SourceExtraction(
        extraction_id="PP_001",
        category="project_participation",
        statement=(
            "General Atomics shows interest in joining the project through a plus-up "
            "to accelerate deployment of fast next-generation online monitoring of "
            "nuclear materials."
        ),
        page=15,
        comparison_state="proposed_participation",
    ),
    SourceExtraction(
        extraction_id="PP_002",
        category="project_participation",
        statement="The project is seeking a plus-up to add General Atomics to the project team.",
        page=16,
        comparison_state="proposed_participation",
    ),
    SourceExtraction(
        extraction_id="TS_001",
        category="technical_scope",
        statement="Solve the absorber manufacturing problem.",
        page=15,
        comparison_state="plus_up_technical_scope",
    ),
    SourceExtraction(
        extraction_id="TS_002",
        category="technical_scope",
        statement="Build up General Atomics microcalorimeter expertise.",
        page=15,
        comparison_state="plus_up_technical_scope",
    ),
    SourceExtraction(
        extraction_id="BE_001",
        category="business_evaluation",
        statement="Evaluate the commercial TES business opportunity.",
        page=15,
        comparison_state="technology_to_market_scope",
    ),
    SourceExtraction(
        extraction_id="PR_001",
        category="product_requirement",
        statement="Define requirements for product assembly.",
        page=15,
        comparison_state="technology_to_market_scope",
    ),
    SourceExtraction(
        extraction_id="DS_001",
        category="deployment_scope",
        statement="Deploy a small 96-pixel module at the INL Analytical Laboratory.",
        page=15,
        metric="module_pixel_count",
        value=96,
        unit="pixels",
        comparison_state="current_scope",
    ),
    SourceExtraction(
        extraction_id="DS_002",
        category="deployment_scope",
        statement="Support measurements in 2 minutes through a 4,000-pixel instrument.",
        page=15,
        metric="instrument_pixel_count",
        value=4000,
        unit="pixels",
        comparison_state="long_term_goal",
    ),
]

for item in EXTRACTIONS:
    item.validate()

len(EXTRACTIONS)


## Review Extracted Transition Content

Inspect this table before generating RP_47 YAML.


In [ ]:
extraction_table = pd.DataFrame(asdict(item) for item in EXTRACTIONS)
extraction_table[
    [
        "extraction_id",
        "category",
        "page",
        "metric",
        "value",
        "unit",
        "statement",
    ]
]


## Technology-Transition Sequence

```text
140 cps Aids Commercialization
        ↓
General Atomics Project Plus-Up
        ↓
Commercial TES Opportunity Evaluation
        ↓
Product Assembly Requirements
```

The sequence records proposed participation and evaluation. It does not claim that General Atomics has joined the project or that a product has been commercialized.


In [ ]:
SEQUENCE_ROWS = [
    {
        "stage": "A",
        "input": "140 cps Aids Commercialization",
        "output": "General Atomics Project Plus-Up",
        "support_a": "Near Target Speed",
        "support_b": "Online Nuclear Monitoring",
    },
    {
        "stage": "B",
        "input": "General Atomics Project Plus-Up",
        "output": "Commercial TES Opportunity Evaluation",
        "support_a": "Absorber Manufacturing",
        "support_b": "GA Microcalorimeter Expertise",
    },
    {
        "stage": "C",
        "input": "Commercial TES Opportunity Evaluation",
        "output": "Product Assembly Requirements",
        "support_a": "96-Pixel INL Module",
        "support_b": "4,000-Pixel Goal",
    },
]

pd.DataFrame(SEQUENCE_ROWS)


## Candidate Source-Derived Reading Point

The visible dialogue uses source-supported transition language. The repository grammar remains in metadata.


In [ ]:
READING_POINT_CANDIDATE = {
    "reading_point_id": "RP_47",
    "engineering_object": "Microcalorimeter Technology Transition",
    "source_id": SOURCE["source_id"],
    "dialogue": [
        {
            "order": "A",
            "concept": "Project Participation",
            "title": "Project Participation: Microcalorimeters",
            "first_label": "140 cps Aids Commercialization",
            "second_label": "General Atomics Project Plus-Up",
            "supporting_context": [
                "Near Target Speed",
                "Online Nuclear Monitoring",
            ],
            "engineering_statement": (
                "Detector progress supports a proposed General Atomics project plus-up."
            ),
        },
        {
            "order": "B",
            "concept": "Commercial Evaluation",
            "title": "Commercial Evaluation: Microcalorimeters",
            "first_label": "General Atomics Project Plus-Up",
            "second_label": "Commercial TES Opportunity Evaluation",
            "supporting_context": [
                "Absorber Manufacturing",
                "GA Microcalorimeter Expertise",
            ],
            "engineering_statement": (
                "Proposed project participation supports evaluation of the commercial "
                "TES business opportunity."
            ),
        },
        {
            "order": "C",
            "concept": "Product Requirements",
            "title": "Product Requirements: Microcalorimeters",
            "first_label": "Commercial TES Opportunity Evaluation",
            "second_label": "Product Assembly Requirements",
            "supporting_context": [
                "96-Pixel INL Module",
                "4,000-Pixel Goal",
            ],
            "engineering_statement": (
                "Commercial opportunity evaluation directs product-assembly requirements."
            ),
        },
    ],
}

READING_POINT_CANDIDATE


## Export Source Record and RP_47 Specifications

The notebook directly generates cumulative `RP_47_A.yaml`, `RP_47_B.yaml`, and `RP_47_C.yaml`.


In [ ]:
source_record = {
    "source": SOURCE,
    "extractions": [asdict(item) for item in EXTRACTIONS],
    "sequence": SEQUENCE_ROWS,
    "reading_point_candidate": READING_POINT_CANDIDATE,
}

json_path = OUTPUT_DIRECTORY / "becker_2025_rp_47_source_extraction.json"
yaml_path = OUTPUT_DIRECTORY / "becker_2025_rp_47_source_extraction.yaml"
review_path = OUTPUT_DIRECTORY / "becker_2025_rp_47_source_extraction.md"
candidate_path = OUTPUT_DIRECTORY / "RP_47_SOURCE_DERIVED.yaml"

json_path.write_text(
    json.dumps(source_record, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
yaml_path.write_text(
    yaml.safe_dump(source_record, sort_keys=False, allow_unicode=True, width=100),
    encoding="utf-8",
)
candidate_path.write_text(
    yaml.safe_dump(
        READING_POINT_CANDIDATE,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    ),
    encoding="utf-8",
)

review_lines = [
    "# RP_47 Source Extraction",
    "",
    f"**Source:** {SOURCE['title']}",
    "",
    "## Technology-Transition Sequence",
    "",
    "```text",
    "140 cps Aids Commercialization",
    "        ↓",
    "General Atomics Project Plus-Up",
    "        ↓",
    "Commercial TES Opportunity Evaluation",
    "        ↓",
    "Product Assembly Requirements",
    "```",
    "",
    "## Source-Derived Extractions",
    "",
]
for item in EXTRACTIONS:
    review_lines.append(f"- Page {item.page}: {item.statement}")

review_lines.extend(
    [
        "",
        "## Scope Boundary",
        "",
        "- General Atomics participation is proposed, not completed.",
        "- Commercial evaluation is a stated scope, not a positive business conclusion.",
        "- Product assembly requirements are to be defined; no finished product is claimed.",
        "",
        "*Admissible generalizations trail leading specifications.*",
    ]
)

review_path.write_text("\n".join(review_lines) + "\n", encoding="utf-8")

REPOSITORY_GRAMMAR = [
    "Engineering Object specifies Engineering System.",
    "Engineering System produces Measured Engineering States.",
    "Measurement records Measured Engineering States.",
    "Measured Engineering States identify Engineering Constraints.",
    "Engineering Constraints direct Engineering Refinements.",
    "Engineering Refinements support Measured Engineering Improvement.",
    "Measured Engineering Improvement informs Leading Specifications.",
    "Leading Specifications direct Engineering Priorities.",
    "Engineering Priorities prepare Engineering Sessions.",
    "Engineering Sessions produce Engineering Records.",
    "Engineering Records support Engineering Reports.",
    "Engineering Reports support Repository Contributions.",
    "Repository Contributions support Repository Development.",
    "Repository Development supports Continued Specification.",
    "Continued Specification supports Engineering Object.",
]

STAGE_CONFIGURATION = {
    "A": {
        "notebook_id": "NB_47_A_PROJECT_PARTICIPATION",
        "inherited_from": "NB_43_C_INSTRUMENT_SCALING",
        "natural_foundation": "Instrument Scaling",
        "engineering_objective": (
            "Specify proposed project participation from source-derived "
            "commercialization progress."
        ),
        "forward_context": "Commercial Evaluation",
        "completion_status": "developing",
    },
    "B": {
        "notebook_id": "NB_47_B_COMMERCIAL_EVALUATION",
        "inherited_from": "NB_47_A_PROJECT_PARTICIPATION",
        "natural_foundation": "Project Participation",
        "engineering_objective": (
            "Specify commercial TES opportunity evaluation from the proposed "
            "General Atomics plus-up scope."
        ),
        "forward_context": "Product Requirements",
        "completion_status": "developing",
    },
    "C": {
        "notebook_id": "NB_47_C_PRODUCT_REQUIREMENTS",
        "inherited_from": "NB_47_B_COMMERCIAL_EVALUATION",
        "natural_foundation": "Commercial Evaluation",
        "engineering_objective": (
            "Specify product-assembly requirements from the source-derived "
            "technology-to-market scope."
        ),
        "forward_context": "Technology Transition",
        "completion_status": "complete",
    },
}


def cumulative_rp_specification(stage: str) -> dict[str, Any]:
    stage_order = {"A": 1, "B": 2, "C": 3}
    if stage not in stage_order:
        raise ValueError(f"Unsupported RP_47 stage: {stage}")

    included_source = READING_POINT_CANDIDATE["dialogue"][: stage_order[stage]]
    included_dialogue = []

    for index, source_item in enumerate(included_source):
        dialogue_stage = source_item["order"]
        dialogue_status = (
            "candidate"
            if index == len(included_source) - 1
            else "admitted"
        )

        included_dialogue.append(
            {
                "order": dialogue_stage,
                "artifact_id": (
                    f"47_{dialogue_stage}_"
                    f"{source_item['concept'].lower().replace(' ', '_')}_trail"
                ),
                "concept": source_item["concept"],
                "title": source_item["title"],
                "first_label": source_item["first_label"],
                "second_label": source_item["second_label"],
                "supporting_context": source_item["supporting_context"],
                "engineering_statement": source_item["engineering_statement"],
                "status": dialogue_status,
            }
        )

    configuration = STAGE_CONFIGURATION[stage]

    return {
        "identity": {
            "notebook_id": configuration["notebook_id"],
            "reading_point": "RP_47",
            "stage": stage,
            "version": "1.0.0",
            "status": "candidate",
        },
        "reading_point": {
            "inherited_from": configuration["inherited_from"],
            "natural_foundation": configuration["natural_foundation"],
            "engineering_objective": configuration["engineering_objective"],
            "engineering_statements": [
                item["engineering_statement"]
                for item in included_dialogue
            ],
            "repository_grammar": REPOSITORY_GRAMMAR,
            "forward_context": configuration["forward_context"],
            "status": configuration["completion_status"],
        },
        "dialogue": included_dialogue,
        "engineering_object": SOURCE["engineering_object"],
        "engineering_direction": SOURCE["engineering_direction"],
        "source": SOURCE,
        "source_engineering_states": {
            category: [
                asdict(item)
                for item in EXTRACTIONS
                if item.category == category
            ]
            for category in {
                "commercialization_indicator",
                "project_participation",
                "technical_scope",
                "business_evaluation",
                "product_requirement",
                "deployment_scope",
            }
        },
        "footer": "Admissible generalizations trail leading specifications.",
    }


rp_paths = {}
rp_specifications = {}

for stage in ("A", "B", "C"):
    specification = cumulative_rp_specification(stage)
    path = OUTPUT_DIRECTORY / f"RP_47_{stage}.yaml"

    path.write_text(
        yaml.safe_dump(
            specification,
            sort_keys=False,
            allow_unicode=True,
            width=100,
        ),
        encoding="utf-8",
    )

    rp_paths[stage] = path
    rp_specifications[stage] = specification

readme_path = OUTPUT_DIRECTORY / "RP_47_README.md"
readme_path.write_text(
    "# RP_47 — Technology-to-Market Scope\n\n"
    "A: detector progress → proposed General Atomics project plus-up\n\n"
    "B: proposed project participation → commercial TES opportunity evaluation\n\n"
    "C: commercial opportunity evaluation → product assembly requirements\n\n"
    "Generated directly by NB_00_RP_47_SOURCE_EXTRACTION.\n",
    encoding="utf-8",
)

zip_path = OUTPUT_DIRECTORY / "NB_00_RP_47_SOURCE_EXTRACTION.zip"
bundle_paths = [
    json_path,
    yaml_path,
    review_path,
    candidate_path,
    rp_paths["A"],
    rp_paths["B"],
    rp_paths["C"],
    readme_path,
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in bundle_paths:
        archive.write(path, arcname=path.name)

generated = {
    "source_json": json_path,
    "source_yaml": yaml_path,
    "source_review": review_path,
    "reading_point_candidate": candidate_path,
    "rp_47_a": rp_paths["A"],
    "rp_47_b": rp_paths["B"],
    "rp_47_c": rp_paths["C"],
    "rp_47_readme": readme_path,
    "zip": zip_path,
}

generated


## Verification

This final cell verifies cumulative structure, exact titles, inheritance, source provenance, statement agreement, and the proposed-status scope boundaries.


In [ ]:
expected_dialogue_counts = {"A": 1, "B": 2, "C": 3}

expected_titles = {
    "A": "Project Participation: Microcalorimeters",
    "B": "Commercial Evaluation: Microcalorimeters",
    "C": "Product Requirements: Microcalorimeters",
}

expected_inheritance = {
    "A": "NB_43_C_INSTRUMENT_SCALING",
    "B": "NB_47_A_PROJECT_PARTICIPATION",
    "C": "NB_47_B_COMMERCIAL_EVALUATION",
}

for label, path in generated.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    if path.stat().st_size <= 0:
        raise ValueError(f"Empty {label}: {path}")

for stage, expected_count in expected_dialogue_counts.items():
    loaded = yaml.safe_load(rp_paths[stage].read_text(encoding="utf-8"))

    identity = loaded["identity"]
    reading_point = loaded["reading_point"]
    dialogue = loaded["dialogue"]

    if identity["reading_point"] != "RP_47":
        raise ValueError(f"RP_47_{stage} has the wrong Reading Point identity")

    if identity["stage"] != stage:
        raise ValueError(f"RP_47_{stage} stage mismatch")

    if len(dialogue) != expected_count:
        raise ValueError(
            f"RP_47_{stage} contains {len(dialogue)} dialogues; "
            f"expected {expected_count}"
        )

    if dialogue[-1]["title"] != expected_titles[stage]:
        raise ValueError(f"RP_47_{stage} title mismatch")

    if reading_point["inherited_from"] != expected_inheritance[stage]:
        raise ValueError(f"RP_47_{stage} inheritance mismatch")

    if reading_point["engineering_statements"] != [
        item["engineering_statement"]
        for item in dialogue
    ]:
        raise ValueError(f"RP_47_{stage} statement mismatch")

    if loaded["source"]["source_id"] != SOURCE["source_id"]:
        raise ValueError(f"RP_47_{stage} source identity was not preserved")

all_dialogue_text = json.dumps(
    READING_POINT_CANDIDATE,
    ensure_ascii=False,
).lower()

for prohibited_claim in (
    "general atomics joined",
    "commercial agreement",
    "manufacturing partner",
    "commercialized product",
):
    if prohibited_claim in all_dialogue_text:
        raise ValueError(
            f"Unsupported completed-state claim detected: {prohibited_claim}"
        )

print("RP_47 source extraction and YAML bundle: VERIFIED")
for label, path in generated.items():
    print(f"{label}: {path} ({path.stat().st_size} bytes)")

print()
print("Next:")
print("1. Extract RP_47_A.yaml, RP_47_B.yaml, and RP_47_C.yaml from the ZIP.")
print("2. Use each file as templates/RP_TEMPLATE.yaml.")
print("3. Run NB_TEMPLATE.ipynb for each cumulative stage.")

try:
    from google.colab import files
except ModuleNotFoundError:
    pass
else:
    files.download(str(zip_path))
